# Build a Tool-Use ReAct Agentic AI System with LangGraph from Scratch


## Problem Statement

We aim to build **HealthBuddy**, an Agentic AI system designed to assist users with health-related queries through intelligent reasoning and tool use. Leveraging the **ReAct framework** (Reasoning + Acting), HealthBuddy integrates a **Large Language Model (LLM)** with custom tools and structured instructions to provide accurate, actionable, and personalized responses.

HealthBuddy is capable of:

- Answering health-related queries, including symptom checks and treatment suggestions, by using external sources such as **Web Search** and **PubMed Search** tools
- Recommending appropriate doctors for specific medical concerns via a **Doctor Recommendation Tool**
- Displaying available **appointment slots** and **booking appointments** after collecting essential user details (name, email, phone)
- Maintaining **multi-turn conversations** in a natural, chat-like flow
- Supporting **multi-user interactions** with isolated memory contexts for each user to enable personalized and continuous engagement

Each notebook in this series will build and extend different components of the system, progressively enhancing HealthBuddy’s intelligence, tool-use capabilities, and user experience.


### Objective:

Our goals in this notebook are:

- Build a ReAct-based Tool-Use Agent using LangGraph from scratch

- Equip the agent with multiple tools (web search, PubMed search, doctor recommendation)

- Learn how to manage agent state using the State Graph

- Learn how to create nodes, edges, and connect them into a powerful ReAct agentic workflow

- Simulate a multi-step reasoning and tool-using workflow

By the end of this notebook, we will have a working agent that can research health queries, gather relevant information, and offer advice—including suggesting a doctor—through structured decision-making and tool usage.

### Agent Architecture:

The figure below shows the agent architecture, including all components and the overall workflow:


In [ ]:
from IPython.display import Image
Image(filename='/Users/csharm33/code/genai_dbx/Course4/Data/tool_use_agent.png')


**Note:** We will build the same agent from the previous demo but now construct it entirely from scratch (with nodes, edges, etc.). This demonstrates how the agent can be customized, particularly in the next live demo where we make it conversational.


In [ ]:
#%run /Users/csharm33/code/genai_dbx/Course3/.setup/learner_setup.ipynb

## Load Necessary Dependencies


In [ ]:
import json
from langchain_openai import AzureChatOpenAI
from langchain_openai import AzureOpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain_core.tools import tool
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from IPython.display import display, Image, Markdown


## Setup Authentication and LLM Client

The following code sets up the UAIS environment to authenticate the application and securely connect to the Azure OpenAI services. It retrieves the access token required for authorization and initializes both the LLM and embedding clients using Azure OpenAI authentication.


In [ ]:
import os
from dotenv import load_dotenv
import httpx

def get_access_token():
    auth = "https://api.uhg.com/oauth2/token"
    scope = "https://api.uhg.com/.default"
    grant_type = "client_credentials"


    with httpx.Client() as client:
        body = {
            "grant_type": grant_type,
            "scope": scope,
            "client_id": client_id,
            "client_secret": client_secret,
        }
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        resp = client.post(auth, headers=headers, data=body, timeout=60)
        access_token = resp.json()["access_token"]
        return access_token


load_dotenv('/Users/csharm33/code/genai_dbx/Course4/Data/vars.env')


AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
OPENAI_API_VERSION = os.environ["OPENAI_API_VERSION"]
EMBEDDINGS_DEPLOYMENT_NAME = os.environ["EMBEDDINGS_DEPLOYMENT_NAME"]
MODEL_DEPLOYMENT_NAME = os.environ["MODEL_DEPLOYMENT_NAME"]
PROJECT_ID = os.environ['PROJECT_ID']
client_id = os.environ.get("CLIENT_ID")
client_secret = os.environ.get("CLIENT_SECRET")

chat_client = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=OPENAI_API_VERSION,
    azure_deployment = MODEL_DEPLOYMENT_NAME,
    temperature=0,
    azure_ad_token=get_access_token(),
    default_headers={
        "projectId": PROJECT_ID
    }
)


embeddings_client = AzureOpenAIEmbeddings(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=OPENAI_API_VERSION,
    azure_deployment=EMBEDDINGS_DEPLOYMENT_NAME,
    azure_ad_token=get_access_token(),
    default_headers={
        "projectId": PROJECT_ID
    }
)


## Prepare Databases for Simulated Web Search and PubMed Search Tools

Here, we will load sample data simulating real web page documents and research papers from PubMed.

Our tools, which we will define later, will reference this data instead of making live internet searches (due to environment and security restrictions preventing internet access).

This simulates real-time web or research paper searches using preloaded documents as the source.

We will create two separate vector databases:

- `web_search_db`: Contains general-purpose health-related web articles, simulating responses to real-time internet queries like “treatments for diabetes” or “how to handle panic attacks”

- `pubmed_db`: Built using sample abstracts and content from medical research papers, simulating PubMed search results for clinically accurate, research-backed information

Later as we proceed in the notebook, agent tools will query these databases to retrieve relevant information and help the LLM provide informed, grounded responses.


In [ ]:
# Loading pre-built datasets for web search and PubMed
with open('/Users/csharm33/code/genai_dbx/Course4/Data/search_data.json', 'r') as f:
    search_docs = json.load(f)

# Display the top-level keys in the loaded dataset
print(f"Major Document Types: {list(search_docs.keys())}")
tiktoken_cache_dir = os.path.abspath("./.setup/tiktoken_cache/")
os.environ["TIKTOKEN_CACHE_DIR"] = tiktoken_cache_dir
os.environ["ANONYMIZED_TELEMETRY"] = "False"

# Create a vector database from simulated web search documents.
# Enables semantic search over general health-related content.
web_search_db = Chroma.from_texts(
    search_docs['search_pages'],              # List of web-style document texts
    collection_name='web_search_db_demo3',    # Collection name
    embedding=embeddings_client,              # Embedding model for vectorization
)

# Create a vector database from simulated PubMed documents.
# Enables semantic search over clinical research content.
pubmed_db = Chroma.from_texts(
    search_docs['pubmed_pages'],              # List of PubMed-style document texts
    collection_name='pubmed_db_demo3',        # Collection name
    embedding=embeddings_client,              # Embedding model for vectorization
)


## Preparing Database for Doctor Recommendations Tool

In this section, we will create a small, in-memory database that contains information about doctors. This data will be used by our **Doctor Recommendation Tool**, which will help users find the right doctor based on their health query or symptoms.

The database includes a list of doctors along with their:
- Name
- Specialization (e.g., Dermatology, Pediatrics, Cardiology)
- Location
- Availability
- Contact information

We are using a simple Python list of dictionaries to store the doctor data. This keeps it easy to understand and modify. In a real-world application, this would typically be replaced by a backend database like PostgreSQL, MongoDB, or an external API.

We will build a tool later on to use this data to recommend doctors based on the user's needs — for example, recommending a pediatrician for a child’s fever or a cardiologist for chest pain.


In [ ]:
# Load doctor dataset with availability, specialization, and contact details
doctors_db = [
    {"name": "Dr. Janet Dyne", "specialization": "Endocrinology (Diabetes Care)", "available_timings": "10:00 AM - 1:00 PM", "location": "City Health Clinic", "contact": "janet.dyne@healthclinic.com"},
    {"name": "Dr. Don Blake", "specialization": "Cardiology (Heart Specialist)", "available_timings": "2:00 PM - 5:00 PM", "location": "Metro Cardiac Center", "contact": "don.blake@metrocardiac.com"},
    {"name": "Dr. Susan D'Souza", "specialization": "Oncology (Cancer Care)", "available_timings": "11:00 AM - 2:00 PM", "location": "Hope Cancer Institute", "contact": "susan.dsouza@hopecancer.org"},
    {"name": "Dr. Matt Murdock", "specialization": "Psychiatry (Mental Health)", "available_timings": "4:00 PM - 7:00 PM", "location": "Mind Care Center", "contact": "matt.murdock@mindcare.com"},
    {"name": "Dr. Dinah Lance", "specialization": "General Physician", "available_timings": "9:00 AM - 12:00 PM", "location": "Downtown Medical Center", "contact": "dinah.lance@downtownmed.com"}
]


## Create Tools for AI Agent

In this section, we will define the tools that our AI Agent will use to perform specific tasks.

LangChain makes it easy to create and register tools using the `Tool` class. A tool includes:
- A name and description
- The python function to be called
- An input schema that tells the model what arguments it can use

When tools are properly defined, they enable the model to solve more complex problems by allowing it to perform actions and access external data. This makes the system more useful and reliable.

### 🧪 Example
```python


In [ ]:
from langchain.tools import tool

# Tool for simulating a web search on general health topics
@tool
def search_web(query: str) -> list:
    """Search the web for a query. Useful for retrieving general or up-to-date healthcare information."""
    # Perform semantic similarity search over the web search vector database
    results = web_search_db.similarity_search(query, k=5)
    docs = [doc.page_content for doc in results]
    return docs


# Tool for simulating a PubMed-style academic literature search
@tool
def search_pubmed(query: str) -> list:
    """Search PubMed for scientific articles related to the query."""
    # Perform semantic similarity search over the PubMed vector database
    results = pubmed_db.similarity_search(query, k=5)
    docs = [doc.page_content for doc in results]
    return docs


# Tool for recommending a doctor based on user symptoms or health-related queries
@tool
def recommend_doctor(query: str) -> dict:
    """Recommend the most suitable doctor based on the user's symptoms."""
    doctors_list = str(doctors_db)

    # Use the LLM to reason over the list and identify the best match for the user's concern
    prompt = f"""You are an assistant helping recommend a doctor based on a patient's health issues.

Here is the list of available doctors:
{doctors_list}

Given the user's query: "{query}"

Choose the most suitable doctor from the list. Only pick one doctor.
Return only the selected doctor's information in JSON format.
If unsure, recommend the General Physician.
"""

    response = chat_client.invoke(prompt)
    return {"recommended_doctor": response.content}


## Define Tools to be Used by the Agent

In this section, we define the tools that our AI agent will use when reasoning through user queries. These include:

- `search_web`: To fetch general health-related information from the simulated web vector store
- `search_pubmed`: To retrieve scientific insights from the simulated PubMed research paper store
- `recommend_doctor`: To recommend a suitable doctor based on the user's symptoms

These tools are already defined and registered using the `@tool` decorator. Here, we simply collect them into a list and bind them to the LLM using LangChain’s `bind_tools()` method. This gives the LLM the ability to request tool usage during its reasoning process.

# List of all tools that the LLM should be aware of


In [ ]:
# These tools were defined earlier using the @tool decorator
tools = [search_web, search_pubmed, recommend_doctor]

# Bind the tools to the LLM so it can invoke them when necessary
# Enables tool-calling functionality in LangChain
llm_with_tools = chat_client.bind_tools(tools=tools)


## Define Agent Instructions Prompt

To guide the LLM-based agent, we provide a custom system prompt that sets the role and behavior of the assistant.

The prompt clearly instructs the agent to:
- Act as a helpful healthcare assistant
- Reason on input queries and do the following:
  - Research the query using the most relevant tools (web, pubmed)
  - Recommend a doctor only if appropriate

This prompt plays a critical role in shaping how the agent reasons, decides when to call tools, and how to construct final responses. It supports ReAct-style behavior where the model reflects, takes actions (via tools), and continues reasoning.


In [ ]:
# Instruction prompt for the overall Agent
AGENT_PROMPT_TXT = """You are an agent designed to act as an expert in researching medical symptoms
and recommending relevant doctors for booking appointments. Also, remember the current year is 2025.

Given a user query, call the relevant tools and provide the most appropriate response.
Follow these guidelines to help you make more informed decisions:
  - If the user's query specifically asks for a doctor recommendation, recommend an appropriate doctor.
  - If the user is researching specific aspects related to symptoms, treatments, or other areas of healthcare,
    use both the search_web and search_pubmed tools to gather detailed information and provide a well-structured response.
  - If the user is just looking for general healthcare information, web search alone is sufficient.
  - Use the search_pubmed tool only if the query relates to information typically found in PubMed articles.
  - Responses should include cited source links and/or PubMed article titles and publication dates, if available.
  - If recommending doctors, use the recommend_doctor tool and display detailed information in a clear, structured format.
    Also, encourage the user to book an appointment via email.
  - Politely decline to answer any queries that are not related to medical or healthcare information.

The final response should contain the following:
- The main output content.
- At the end, include an Agent Reasoning: section that covers the following in bullet points:
    - Your step-by-step reasoning process for arriving at the final response.
    - The tools you called, in the specific order, with their names.
    - Observations from the tool call results that helped you construct the final response.
"""

AGENT_SYS_PROMPT = SystemMessage(content=AGENT_PROMPT_TXT)


## Define Agent State Schema

In LangGraph, an agent’s state can be represented as a dictionary with various keys and values. Here we maintain state using a structured list of messages.

This state defines what data is carried forward between each step in the agent's reasoning and decision-making process.

Here, we define a simple state using Python’s `TypedDict` with one key field:

- `messages`: A list of all messages in the Agent so far. This includes the user query, LLM responses, any tool call requests, and the tool responses

The `messages` list is annotated using `add_messages`, which ensures that every new message is appended properly as the agent moves through each step in the graph.

This schema is later passed to the LangGraph `StateGraph`, enabling the agent to track all reasoning and tool usage as a growing message history.

It forms the foundation for the ReAct loop — Reason, Act (tool call), Observe, Repeat — that we build using LangGraph.


In [ ]:
# Define the agent's state schema for storing the message history
class State(TypedDict):
    messages: Annotated[list, add_messages]


## Create our Tool-Use AI Agent from Scratch

In this section, we build the full ReAct-style agent manually using `StateGraph` from LangGraph.

Here’s what we do step-by-step:

1. **Define the LLM reasoning step**:
   - `tool_calling_llm()` is the function used to invoke the LLM at each step
   - It adds the system prompt (`AGENT_SYS_PROMPT`) to the current message history and queries the LLM using `llm_with_tools.invoke(...)`
   - The response (which may be a tool call request or final answer) is returned and stored as the updated `messages`

2. **Initialize the graph**:
   - A `StateGraph` is built using our `State` schema

3. **Add nodes**:
   - `"tool_calling_llm"` is the planning and reasoning node where the agent decides what to do next
   - `"tools"` is a `ToolNode`, which handles execution of whichever tool the LLM requested

4. **Add routing logic**:
   - We start the graph from `"tool_calling_llm"`
   - We use `add_conditional_edges(...)` to route based on whether the LLM output is a tool call request using the `tools_condition` utility function:
     - If it is a tool call request, we go to the `tools` node to execute the requested function
     - If not, we route to `END`, completing the agent's task

This section gives us full control over the agent’s behavior — defining exactly how reasoning, tool use, and control transitions work, without relying on black-box abstractions.


In [ ]:
# Create the node function that handles reasoning and planning using the LLM
def tool_calling_llm(state: State) -> State:
    # Extract the current conversation history from the state
    current_state = state["messages"]

    # Prepend the system instructions to the current message history
    state_with_instructions = [AGENT_SYS_PROMPT] + current_state

    # Call the LLM to generate a new message (either a response or a tool call request)
    response = [llm_with_tools.invoke(state_with_instructions)]

    # Return the updated state containing the new message
    return {"messages": response}

# Build the graph
builder = StateGraph(State)

# Add nodes
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools=tools))

# Add edges
builder.add_edge(START, "tool_calling_llm")
# Conditional edge
builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition, # conditional routing function
    # If the latest message (result) from LLM is a tool call request -> tools_condition routes to tools
    # If the latest message (result) from LLM is a not a tool call -> tools_condition routes to END
    ["tools", END]
)
builder.add_edge("tools", "tool_calling_llm") # this is the key feedback loop in the agentic system

# Compile Agent Graph
healthbuddy_agent = builder.compile()


## View our Agent Flow

This is what the agent workflow looks like.


In [ ]:
from IPython.display import Image
Image(filename='/Users/csharm33/code/genai_dbx/Course4/Data/tool_use_agent_arch.png', height=200, width=300)


## Build Utility Function to Stream Agent Results

We define a helper function to stream the step-by-step output of the agent. This makes it easier to trace:
- What the agent doing in each step
- Which tool it decides to call
- What response it gets from that tool
- How it forms the final reply

Streaming output is helpful when evaluating multi-step reasoning and debugging tool use in real time.


In [ ]:
# Utility function to call the agent and stream its step-by-step reasoning
def call_agent(agent, query, verbose=False):

    # Stream the agent's execution for the given query
    for event in agent.stream(
        {"messages": [HumanMessage(content=query)]},  # Input prompt
        stream_mode='values'  # Stream output as intermediate values
    ):
        # If verbose is enabled, print each intermediate message
        if verbose:
            event["messages"][-1].pretty_print()

    # Display the final response from the agent as Markdown
    print('\n\nFinal Response:\n')
    display(Markdown(event["messages"][-1].content))

    # Return the final message content for optional downstream use
    return event["messages"][-1].content


## Test out our Agent!

In this final section, we run a complete test of our Tool-Use AI Agent by passing it a sample health-related query.

We observe how the agent:
- Interprets the query
- Decides which tool(s) to use (if any)
- Executes tool calls
- Streams the intermediate steps and final output


In [ ]:
# Example usage
query = "what are the latest methods for diabetes management and recommend a doctor please"
result = call_agent(healthbuddy_agent, query, verbose=True)
# Example usage
query = "I am having panic attacks, what could I do?"
result = call_agent(healthbuddy_agent, query, verbose=True)
# Example usage without printing detailed log messages
query = "I am having panic attacks, please recommend a right doctor"
result = call_agent(healthbuddy_agent, query, verbose=False)


## Agent Limitations

- Can't recall prior conversation
- Needs memory integration to be conversational


In [ ]:
query = "Great can you book an appointment please"
result = call_agent(healthbuddy_agent, query, verbose=True)


## Homework: Enhance the ReAct Agent with a New Capability
**Goal:** Extend the HealthBuddy ReAct agent by adding a new healthcare-specific tool (building on tools you’ve already implemented) and integrating it into the agent’s workflow using LangGraph.


#### Tasks
1. **1.	Add a New Healthcare Tool:**  Choose one of the following or come up with your own::
    - BMI Calculator: Computes BMI and provides weight classification
    - Medication-to-Condition Lookup: Maps common medications to conditions they treat
    - Symptom Checker: Suggests possible causes for common symptoms


2. **Example:** To implement the BMI Calculator, write a custom python function.  Use the formula :


    $BMI = \frac{weight (kg)}{height (m)^2}$

   and return a category:
    - Below 18.5 → Underweight
    - 18.5 to 24.9 → Normal
    - 25 to 29.9 → Overweight
    - 30 and above → Obese


3. **Register the Tool with the LLM:** Decorate your tool function with @tool and add it to the tools list alongside existing ones like:
    - web_search_tool
    - pubmed_tool
    - doctor_recommender_tool



4. **Improve the System Instruction Prompt:**  
    - Update the system prompt used in the agent to include when and how the new tool should be used.
    - Give clear, natural language guidance for invoking the tool, e.g.:
        “Use the BMI Calculator when users ask about weight or body health based on height and weight.”
5. **Update the Agent Configuration:** Modify the create_react_agent() call to include your tool. Ensure it integrates smoothly with other tools in reasoning workflows.

6. **Test End-to-End Flow:** Try at least two realistic queries that invoke the new tool and produce helpful responses, such as:
    - “What’s the BMI for someone who weighs 68 kg and is 165 cm tall?”
    - “Tell me if someone at 95 kg and 1.8 meters height is in the healthy weight range.”
